# EE 243 — Assignment 3, Problem 1

**Homography estimation (3 pts)** — Checkerboard images **2** and **9**.

Estimate **H** such that P_9 = H P_2 using `cv2.findChessboardCorners` for corners only.

> Do **not** use built-in homography estimators (`findHomography`, etc.). Implement DLT yourself.

In [ ]:
# Setup — run locally or in Colab
!pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Place checkerboard images in this folder (download from course DATASETS drive)
DATA_DIR = Path('.')
IMG2_PATH = DATA_DIR / 'checkerboard_2.png'   # TODO: match your filename
IMG9_PATH = DATA_DIR / 'checkerboard_9.png'   # TODO: match your filename

# Comment out asserts until images are downloaded:
# assert IMG2_PATH.exists(), f'Missing {IMG2_PATH} — add image to Problem1/'
# assert IMG9_PATH.exists(), f'Missing {IMG9_PATH} — add image to Problem1/'


In [ ]:
def detect_checkerboard_corners(image_path, pattern_size=(9, 6)):
    """Return (N, 2) pixel coordinates of inner chessboard corners."""
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ok, corners = cv2.findChessboardCorners(gray, pattern_size, None)
    if not ok:
        raise RuntimeError(f'Chessboard not found in {image_path}')
    corners = cv2.cornerSubPix(
        gray, corners, (11, 11), (-1, -1),
        criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001),
    )
    return corners.reshape(-1, 2), img


def homography_dlt(src_pts, dst_pts):
    """Estimate 3x3 homography H with dst ~ H src (homogeneous). No cv2.findHomography.

    Args:
        src_pts: (N, 2) points in image 2
        dst_pts: (N, 2) corresponding points in image 9
    Returns:
        H: (3, 3) homography
    """
    # TODO: Build Ah = 0 system (DLT), solve with SVD, normalize H[-1,-1] = 1
    raise NotImplementedError


def apply_homography(H, pts):
    """Map (N, 2) points through homography H."""
    pts_h = np.hstack([pts, np.ones((len(pts), 1))])
    mapped = (H @ pts_h.T).T
    return mapped[:, :2] / mapped[:, 2:3]


In [ ]:
# TODO: set pattern_size to match your checkerboard (cols-1, rows-1 inner corners)
PATTERN_SIZE = (9, 6)

pts2, img2 = detect_checkerboard_corners(IMG2_PATH, PATTERN_SIZE)
pts9, img9 = detect_checkerboard_corners(IMG9_PATH, PATTERN_SIZE)

assert pts2.shape == pts9.shape, 'Corner counts must match (one-to-one correspondence)'

H = homography_dlt(pts2, pts9)
print('Homography H (image 2 -> image 9):')
print(H)

pts9_hat = apply_homography(H, pts2)
rmse = np.sqrt(np.mean((pts9_hat - pts9) ** 2))
print(f'Corner reprojection RMSE: {rmse:.4f} px')


In [ ]:
# Visualize correspondences (optional, for report)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img, pts, title in zip(axes, [img2, img9], [pts2, pts9], ['Image 2', 'Image 9']):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.scatter(pts[:, 0], pts[:, 1], s=20, c='lime', edgecolors='k', linewidths=0.3)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
